# 실습 프로젝트 : 회귀모델 정확도 향상 프로젝트

## 실습 목표
현재 구현된 PyTorch 회귀 모델의 성능을 분석하고 다양한 방법으로 정확도를 향상시켜 보자.

In [7]:
# 문제 1. 현재 모델 성능 확인 ==========================================
# 현재 코드의 평가 지표를 확인하시오.

# 다음 값을 출력하시오.
# * MSE
''' 
MSE는 mean squared error 평균제곱 오차라한다.
정담 - 예측값을 제곱해서 모두 더한 뒤 평균을 낸다 .
오차를 제곱을 하니 틀린 정도가 클수록 더 큰 패널티를 주는 지표
'''
# * RMSE
''' 
RMSE는 root mean squared error 평균제곱근 오차라한다
MSE가 제곱을 해서 단위가 바뀌어버리는 단점을 보완하기 위해 루트를 씌운값이다.
'''
# * MAE
''' 
MAE는 mean absolute error 평균절대 오차라한다.
정답 - 예측값의 절대값을 모두 더한 뒤 평균을 낸
'''
# * R² Score
''' 
R² Score는 결정계수라한다.
오차의 양이 아니라 이 모델이 얼마나 쓸만한가를 0에서 1사이의 숫자로 보여준다고 한다.
'''

### 코드
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score
import numpy as np

# [SH] start
import torch
import torch.nn as nn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
df = pd.read_csv(url, sep=';')

X = df.drop('quality', axis=1).values
y = df['quality'].values.reshape(-1, 1)
feature_names = df.columns[:-1]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.FloatTensor(y_test)
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)


class WineQualityRegressor(nn.Module):
    def __init__(self, input_dim):
        super(WineQualityRegressor, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.network(x)

input_dim = X_train_t.shape[1]
model = WineQualityRegressor(input_dim)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

model.train()
for epoch in range(200):
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()
        optimizer.step()

model.eval()
with torch.no_grad():
    predictions = model(X_test_t).numpy()

# [SH] end
mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)
print("MSE :", mse)
print("RMSE :", rmse)
print("MAE :", mae)
print("R2 :", r2)

## 질문에 대해 답을 작성하시오.  ------------------------------------------------------------
# 1. RMSE의 의미는?
'''
위에서 이미 서치해서 공부해버렸는데..재기록.
'''

''' 
RMSE는 root mean squared error 평균제곱근 오차라한다
MSE가 제곱을 해서 단위가 바뀌어버리는 단점을 보완하기 위해 루트를 씌운값이다.
'''

# 2. MAE의 의미는?
''' 
MAE는 mean absolute error 평균절대 오차라한다.
정답 - 예측값의 절대값을 모두 더한 뒤 평균을 낸
'''

# 3. R² 값이 1에 가까울수록 좋은 이유는?
''' 
R² Score는 결정계수라한다.
오차의 양이 아니라 이 모델이 얼마나 쓸만한가를 0에서 1사이의 숫자로 보여준다고 한다. 
0은 평균값이나 다름없다. 
'''

MSE : 0.35385578870773315
RMSE : 0.5948577886417334
MAE : 0.47362279891967773
R2 : 0.4585269093513489


' \nR² Score는 결정계수라한다.\n오차의 양이 아니라 이 모델이 얼마나 쓸만한가를 0에서 1사이의 숫자로 보여준다고 한다. \n0은 평균값이나 다름없다. \n'